In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
import torch
import gc
from IPython.display import display

ds_name = 'markov_32feat_11t5g_more_balanced_largeN_censor'
ds_df_dir = '/home/yc366/repos/survsurf_benchmark/dataset_split'
proj_name = 'SurvSurfBenchmark_Markov_censored'
g_resol = 1
split='test'

In [ ]:
DEVICE = 'cpu'
if DEVICE == 'gpu':
    assert torch.cuda.is_available()
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
else:
    device = 'cpu'

# Import and instantiate model

In [ ]:
from dataset_11t5g_markov import DataModuleMarkovSurvSurf, DatasetMarkovSurvSurf

In [ ]:
ds = DatasetMarkovSurvSurf(
    df_dir=ds_df_dir, 
    ds_name=ds_name, 
    g_resol=g_resol, 
    split=split, 
    mode='first_cross_obs_only', 
    separate_g_from_feats=False,
)
df_obs = ds._get_df_Xy_trans_obs()
df_true_prob = ds._get_df_Xy_true_prob()
df_true_prob.loc[
    df_true_prob[ds.colname_g] == 6,:
]

In [ ]:
import pickle
def pred_from_sksurv_joint(run_id, proj_name, split='val'):
    path_pickle = f'./sksurv_models/{proj_name}/Joint/{run_id}.pickle'
    with open(path_pickle, 'rb') as f:
        model = pickle.load(f)
    

    ds = DatasetMarkovSurvSurf(
        df_dir=ds_df_dir, 
        ds_name=ds_name, 
        g_resol=g_resol, 
        split=split, 
        mode='first_cross_obs_only', 
        separate_g_from_feats=False,
    )
    df_obs = ds._get_df_Xy_trans_obs()
    df_true_prob = ds._get_df_Xy_true_prob()
    assert (
        df_true_prob.groupby(['subject',ds.colname_g]).apply(
            lambda x: tuple(x['duration']),
            include_groups=False
        )
    ).nunique() == 1


    cols_X = np.r_[[ds.colname_g], df_obs.columns[df_obs.columns.str.startswith('feat')]]      

    df_val_results_obs = []
    df_val_results_grid = []
    for g, df_obs_g in df_obs.groupby(ds.colname_g):
        if g > 5:
            continue
        df_true_prob_g = df_true_prob.loc[
            df_true_prob[ds.colname_g] == g,:
        ]

        n_unique_time_per_subj = (
            df_true_prob_g.groupby(['subject']).apply(
                lambda x: tuple(x['duration'].sort_values()),
                include_groups=False
            )
        ).nunique()
        assert n_unique_time_per_subj == 1, n_unique_time_per_subj
        curvs = model.predict_survival_function(df_obs_g[cols_X])
                
        
        for subj, curv in zip(df_obs_g['subject'], curvs):
            obs_g_subj = df_obs_g.loc[df_obs_g['subject'] == subj,:]
            assert obs_g_subj.shape[0] == 1
            obs_g_subj = obs_g_subj.iloc[0,:]
            t = obs_g_subj['duration']
            truth = obs_g_subj['event_observed']
            if t > curv.x.max():
                pred = curv(curv.x.max())
            else:
                pred = curv(t)
            row = {
                'subj':subj,
                'g':g,
                't':t,
                'pred':1-pred,
                'truth':truth
            }
            df_val_results_obs.append(row)

        
        df_true_prob_g_tless = df_true_prob_g.drop(columns=['duration','event_observed']).drop_duplicates()
        assert df_true_prob_g_tless['subject'].value_counts().max() == 1
        curvs = model.predict_survival_function(df_true_prob_g_tless[cols_X])

        for subj, curv in zip(df_true_prob_g_tless['subject'], curvs):
            df_val_subj_g = pd.DataFrame()
            df_true_prob_subj_g = df_true_prob_g.loc[
                df_true_prob_g['subject'] == subj,:
            ].sort_values('duration')
            t_grid = df_true_prob_subj_g['duration']
            t_grid_in_range = t_grid[t_grid <= curv.x.max()]
            t_grid_extrap = t_grid[t_grid > curv.x.max()]
            pred_prob_at_time_grid = curv(t_grid_in_range)
            pred_prob_at_time_grid = np.r_[pred_prob_at_time_grid, [pred_prob_at_time_grid[-1]]*t_grid_extrap.size]

            df_val_subj_g['t'] = t_grid
            df_val_subj_g['subj'] = subj
            df_val_subj_g['g'] = g
            df_val_subj_g['pred'] = 1-pred_prob_at_time_grid
            df_val_subj_g['truth'] = df_true_prob_subj_g['event_observed']
            df_val_results_grid.append(df_val_subj_g)

    out_dfs = dict()
    out_dfs['obs'] = pd.DataFrame(df_val_results_obs)
    out_dfs['true_prob'] = pd.concat(df_val_results_grid)
    return out_dfs, f'Joint_{model.__class__.__name__}'

In [ ]:
def pred_from_DeepHit(run_id, proj_name, split='val'):
    import wandb
    from model_factory_deephit import LitModelDeepHit
    torch.set_float32_matmul_precision('medium')

    api = wandb.Api()
    # artifact = api.artifact(f'yichenchen-wings/{proj_name}/model-{run_id}:v0', type='model')
    # checkpoint_dir = artifact.download()
    checkpoint_dir = f'/home/yc366/repos/survsurf_benchmark/runtime_results/{proj_name}/{run_id}/checkpoints/'
    os.listdir(checkpoint_dir)[0]
    #path_checkpoint = os.path.join(checkpoint_dir, 'model.ckpt')
    path_checkpoint = os.path.join(checkpoint_dir, os.listdir(checkpoint_dir)[0])
    model_lit = LitModelDeepHit.load_from_checkpoint(path_checkpoint)
    
    datamodule = DataModuleMarkovSurvSurf(
        df_dir=ds_df_dir, 
        ds_name=ds_name, 
        g_resol=g_resol, 
        separate_g_from_feats=False, 
        train_mode='first_cross_obs_only',
        eval_mode='true_probs_grid',
        batch_size=100, 
        num_workers=4)
    datamodule.setup(stage=split)
    if 'val' in split:
        dataloader_obs, dataloader_true_prob = datamodule.val_dataloader()
    if 'test' in split:
        dataloader_obs, dataloader_true_prob = datamodule.test_dataloader()

    with torch.no_grad():
        torch.cuda.empty_cache()
    gc.collect()

    model_loaded_core = model_lit.model
    model_loaded_core = model_loaded_core.to(device)
    model_loaded_core.eval()

    t_size = model_lit.loss_brier.t_size
    t_res = model_lit.loss_brier.t_res

    out_dfs = dict()
    for key, dataloader in [
        ('obs', dataloader_obs),
        ('true_prob', dataloader_true_prob)
    ]:
        
        out_dfs = dict()
    for key, dataloader in [
        ('obs', dataloader_obs),
        ('true_prob', dataloader_true_prob)
    ]:
        subjs = []
        g = []
        t = []
        pred = []
        truth = []
        with torch.no_grad():
            for batch_, (subj, xs, ts, ys, weight, _) in enumerate(dataloader):
                xs, ts, ys = xs.to(device), ts.to(device), ys.to(device)
                out = model_loaded_core.forward_cif(xs)
                assert out.dim() == 3, f'found output with n dim = {out.dim()}, expecting a 3d tensor.'
                assert out.shape[1] == 1, f'found output with second dim = {out.shape[1]}, expecting 1.'
                out = out.reshape(out.shape[0], out.shape[-1])

                t_matrix = ts.repeat(1, t_size)
                
                t_matrix_ref = torch.linspace(0, t_size-1, steps=t_size)*t_res
                t_matrix_ref = t_matrix_ref.repeat(xs.shape[0], 1)

                selector_where_t = torch.logical_and(t_matrix_ref >= t_matrix,  t_matrix_ref < (t_matrix + t_res))
                assert torch.all(selector_where_t.sum(dim=-1) == 1)
                out = out[selector_where_t].reshape(xs.shape[0])

                subjs += list(subj.cpu().numpy())
                g += list(5*xs.cpu().numpy()[:,-1])
                t += list(ts.cpu().numpy()[:,0])
                truth += list(ys.cpu().numpy()[:,0])
                pred += list(out.cpu().numpy())


        pred = np.array(pred)
        truth = np.array(truth)
        df_val_results = pd.DataFrame()
        df_val_results['subj'] = subjs
        df_val_results['g'] = g
        df_val_results['t'] = t
        df_val_results['pred'] = pred
        df_val_results['truth'] = truth
        for col in ['pred', 'truth']:
            df_val_results[col] = df_val_results[col].astype(float)
        out_dfs[key] = df_val_results
    return out_dfs, model_loaded_core.__class__.__name__


In [ ]:

def pred_from_SurvSurf(run_id, proj_name, split='val'):
    import wandb
    from model_factory_survsurf import LitModelSurvSurf
    torch.set_float32_matmul_precision('medium')

    api = wandb.Api()
    # artifact = api.artifact(f'yichenchen-wings/{proj_name}/model-{run_id}:v0', type='model')
    # checkpoint_dir = artifact.download()
    checkpoint_dir = f'/home/yc366/repos/survsurf_benchmark/runtime_results/{proj_name}/{run_id}/checkpoints/'
    os.listdir(checkpoint_dir)[0]
    #path_checkpoint = os.path.join(checkpoint_dir, 'model.ckpt')
    path_checkpoint = os.path.join(checkpoint_dir, os.listdir(checkpoint_dir)[0])
    model_lit = LitModelSurvSurf.load_from_checkpoint(path_checkpoint)
    
    datamodule = DataModuleMarkovSurvSurf(
        df_dir=ds_df_dir, 
        ds_name=ds_name, 
        g_resol=g_resol, 
        separate_g_from_feats=True, 
        train_mode='first_cross_obs_only',
        eval_mode='true_probs_grid',
        batch_size=100, 
        num_workers=4)
    datamodule.setup(stage=split)
    if 'val' in split:
        dataloader_obs, dataloader_true_prob = datamodule.val_dataloader()
    if 'test' in split:
        dataloader_obs, dataloader_true_prob = datamodule.test_dataloader()

    with torch.no_grad():
        torch.cuda.empty_cache()
    gc.collect()

    model_loaded_core = model_lit.model
    model_loaded_core = model_loaded_core.to(device)
    model_loaded_core.eval()


    out_dfs = dict()
    for key, dataloader in [
        ('obs', dataloader_obs),
        ('true_prob', dataloader_true_prob)
    ]:
        
        subjs = []
        g = []
        t = []
        pred = []
        truth = []
        with torch.no_grad():
            for batch_, (subj, xs, gs, ts, ys, weight, _) in enumerate(dataloader):
                xs, gs, ts, ys = xs.to(device), gs.to(device), ts.to(device), ys.to(device)
                out = model_loaded_core(ts, gs, xs)
                subjs += list(subj.cpu().numpy())
                g += list(5*gs.cpu().numpy()[:,0])
                t += list(ts.cpu().numpy()[:,0])
                truth += list(ys.cpu().numpy()[:,0])
                pred += list(out.cpu().numpy()[:,0])


        pred = np.array(pred)
        truth = np.array(truth)
        df_val_results = pd.DataFrame()
        df_val_results['subj'] = subjs
        df_val_results['g'] = g
        df_val_results['t'] = t
        df_val_results['pred'] = pred
        df_val_results['truth'] = truth
        for col in ['pred', 'truth']:
            df_val_results[col] = df_val_results[col].astype(float)
        out_dfs[key] = df_val_results
    return out_dfs, model_loaded_core.__class__.__name__


In [ ]:


ds_train = DatasetMarkovSurvSurf(
    df_dir=ds_df_dir, 
    ds_name=ds_name, 
    g_resol=g_resol, 
    split='train', 
    mode='true_probs_grid', 
    separate_g_from_feats=False
)
ds_val = DatasetMarkovSurvSurf(
    df_dir=ds_df_dir, 
    ds_name=ds_name, 
    g_resol=g_resol, 
    split='val', 
    mode='true_probs_grid', 
    separate_g_from_feats=False
)
ds_test = DatasetMarkovSurvSurf(
    df_dir=ds_df_dir, 
    ds_name=ds_name, 
    g_resol=g_resol, 
    split='test', 
    mode='true_probs_grid', 
    separate_g_from_feats=False
)

In [ ]:
ds_train._get_df_Xy_trans_obs()

### Define scoring functions

In [ ]:
from model_eval_utils import (
    get_brier_and_auc, 
    get_all_grades_start_end_time, 
    get_integrated_brier_intrvl_imputed, 
    get_mse_vs_theory,
    get_mse_vs_theory_certain_obs
)

In [ ]:
from sksurv.nonparametric import kaplan_meier_estimator
from dataset_11t5g_markov import get_pre_censoring_t
class IPCW:
    def __init__(self, time, S_censor):
        self.time = time
        self.S_censor = S_censor
    def __call__(self,t):
        S_censor = self.S_censor
        # percentile  = np.percentile(S_censor, 10)
        # S_censor[S_censor <= percentile] = percentile
        S = np.interp(t, self.time, S_censor)
        return 1/S

def get_train_ipcw(df_event_g_at_t_obs):
    df_event_g_at_t_obs = df_event_g_at_t_obs
    g_to_censor_curv = dict()
    for g, df_event_t in df_event_g_at_t_obs.groupby('g'):
        df_event_t = df_event_t.sort_values('duration')
        time, S_censor = kaplan_meier_estimator(df_event_t['event_observed'].astype(bool), df_event_t['duration'], reverse=True)
        if 0 not in time:
            time = np.r_[[0.], time]
            S_censor = np.r_[[1.], S_censor]
        g_to_censor_curv[g] = IPCW(time, S_censor)
    return g_to_censor_curv

df_last_obs_time_split = get_pre_censoring_t(df_dir=ds_df_dir,ds_name=ds_name, gs=[1,2,3,4,5], split='val')
df_last_obs_time_split = df_last_obs_time_split.loc[df_last_obs_time_split['g'] <6, :]
df_train_obs_censor_less = df_last_obs_time_split.copy()
df_train_obs_censor_less['event_observed'] = True
df_train_obs_censor_less['duration'] = df_train_obs_censor_less['duration'].astype(float)
dict_ipcw = get_train_ipcw(df_last_obs_time_split)

fig, ax = plt.subplots(1,1,)
for g in [1,2,3,4,5]:
    x = np.linspace(0, 10,10)
    y = dict_ipcw[g](x)
    ax.plot(x, y, label=g),
ax.legend()
ax.set(ylabel='ipcw', xlabel='t')


fig, ax = plt.subplots(1,1,)
for g in [1,2,3,4,5]:
    x = np.linspace(0, 10,10)
    y = 1/dict_ipcw[g](x)
    print(f'min censoring S(t) for g = {y.min()}')
    ax.plot(x, y, label=g),
ax.legend()
ax.set(ylabel='S(t) for censoring', xlabel='t')

In [ ]:
df_last_obs_time_split.loc[df_last_obs_time_split['event_observed']==1, 'duration'].describe()

### Make predictions

In [ ]:
df_last_obs_time_split_subj = df_last_obs_time_split.loc[
    df_last_obs_time_split['g'] == 5,:
]
time, survival_prob, conf_int = kaplan_meier_estimator(
    ~df_last_obs_time_split_subj["event_observed"], df_last_obs_time_split_subj["duration"], conf_type="log-log"
)
plt.step(time, survival_prob, where="post")
plt.fill_between(time, conf_int[0], conf_int[1], alpha=0.25, step="post")
plt.ylim(0, 1)
plt.ylabel(r"est. probability of survival $\hat{S}(t)$")
plt.xlabel("time $t$")

In [ ]:
if split == 'test':
    obs_full_traj = ds_test._get_df_Xy_full_traj_obs(g0_as_gres=False)
elif split == 'val':
    obs_full_traj = ds_val._get_df_Xy_full_traj_obs(g0_as_gres=False)
else:
    raise NotImplementedError
obs_trans_time_all_g = get_all_grades_start_end_time(obs_full_traj, gs=[1,2,3,4,5])

In [ ]:
max_time = 9

In [ ]:
model_to_metric = []
model_to_prop_t_violated_per_subj = []
model_to_max_violated_per_subj = []
for run_id, pred_fun, descriptor in [
    # ('0t9voxqg',pred_from_SurvSurf,'SurvSurfTaddTG_LossDyDgSumoFL_dydg_trans_and_last_obs'),
    # ('g2xkuktn',pred_from_SurvSurf,'SurvSurfTaddTG_LossDyDgSumoFL_dydg_trans_and_last_obs_sml'),
    # ('eip5kuz2',pred_from_SurvSurf,'SurvSurfTaddTG_LossDyDgSumoFL_dydg'),
    ('im9pl857',pred_from_SurvSurf,'SurvSurfTaddTG_dt_at_trans_dg_at_last'),
    ('bid6gfka',pred_from_SurvSurf,'SurvSurfTaddTG_dt_at_trans_dg_at_last'),
    ('vxstfcoc',pred_from_SurvSurf,'SurvSurfTaddTG_dt_at_trans_dg_at_last'),
    ('vi64eo47',pred_from_SurvSurf,'SurvSurfTaddTG_dt_at_trans_dg_at_last'),
    ('70v4apjy',pred_from_SurvSurf,'SurvSurfTaddTG_dt_at_trans_dg_at_last'),
    
    ('hqtlr0p1',pred_from_DeepHit,'LossSumo'),
    ('b44jz3sr',pred_from_DeepHit,'LossSumo'),
    ('oon6kf46',pred_from_DeepHit,'LossSumo'),
    ('dn3vmeua',pred_from_DeepHit,'LossSumo'),
    ('vm7rg5y3',pred_from_DeepHit,'LossSumo'),
    
    ('ezcx8l4x',pred_from_DeepHit,'LossDyDg'),
    ('ngklp7ls',pred_from_DeepHit,'LossDyDg'),
    ('rji1nf4m',pred_from_DeepHit,'LossDyDg'),
    ('fveg5k48',pred_from_DeepHit,'LossDyDg'),
    ('75d5ba44',pred_from_DeepHit,'LossDyDg'),
    
    ('Joint_CoxnetSurvivalAnalysis_seed_10',pred_from_sksurv_joint, 'standard'),
    ('Joint_CoxPHSurvivalAnalysis_seed_10',pred_from_sksurv_joint, 'standard'),

    ('Joint_GradientBoostingSurvivalAnalysis_seed_10',pred_from_sksurv_joint, 'standard'),
    ('Joint_GradientBoostingSurvivalAnalysis_seed_20',pred_from_sksurv_joint, 'standard'),
    ('Joint_GradientBoostingSurvivalAnalysis_seed_30',pred_from_sksurv_joint, 'standard'),
    ('Joint_GradientBoostingSurvivalAnalysis_seed_40',pred_from_sksurv_joint, 'standard'),
    ('Joint_GradientBoostingSurvivalAnalysis_seed_50',pred_from_sksurv_joint, 'standard'),

    ('Joint_RandomSurvivalForest_seed_10',pred_from_sksurv_joint, 'standard'),
    ('Joint_RandomSurvivalForest_seed_20',pred_from_sksurv_joint, 'standard'),
    ('Joint_RandomSurvivalForest_seed_30',pred_from_sksurv_joint, 'standard'),
    ('Joint_RandomSurvivalForest_seed_40',pred_from_sksurv_joint, 'standard'),
    ('Joint_RandomSurvivalForest_seed_50',pred_from_sksurv_joint, 'standard')
]:
    out_model, model_name = pred_fun(run_id=run_id, proj_name=proj_name, split=split)
    
    intrg_brier_ipcw_by_subj, mean_auc_ipcw_by_subj = get_brier_and_auc(out_model, df_last_obs_time_split, ipcw='by_subj', max_time=max_time)
    intrg_brier_ipcw_by_g, mean_auc_ipcw_by_g = get_brier_and_auc(out_model, df_last_obs_time_split, ipcw='by_grade', max_time=max_time)

    intrg_brier_intrvl_imputed_ipcw_by_sbj = get_integrated_brier_intrvl_imputed(obs_trans_time_all_g, out_model, df_last_obs_time_split, ipcw='by_subj', max_time=max_time)
    intrg_brier_intrvl_imputed = get_integrated_brier_intrvl_imputed(obs_trans_time_all_g, out_model, df_last_obs_time_split, ipcw='without_ipcw', max_time=max_time)

    intrg_mse_certain_obs_ipcw_by_sbj = get_mse_vs_theory_certain_obs(obs_trans_time_all_g, out_model, df_last_obs_time_split, ipcw='by_subj', max_time=max_time)
    intrg_mse_certain_obs = get_mse_vs_theory_certain_obs(obs_trans_time_all_g, out_model, df_last_obs_time_split, ipcw='without_ipcw', max_time=max_time)


    model_to_metric.append(
        {
            'run_id':run_id,
            'model_type':model_name,
            'intrg_brier_ipcw_by_subj':intrg_brier_ipcw_by_subj,
            'mean_auc_ipcw_by_subj':mean_auc_ipcw_by_subj,
            'intrg_brier_ipcw_by_g':intrg_brier_ipcw_by_g,
            'mean_auc_ipcw_by_g':mean_auc_ipcw_by_g,
            'intrg_brier_intrvl_imputed_ipcw_by_sbj':intrg_brier_intrvl_imputed_ipcw_by_sbj,
            'intrg_brier_intrvl_imputed':intrg_brier_intrvl_imputed,
            'intrg_mse_certain_obs_ipcw_by_sbj':intrg_mse_certain_obs_ipcw_by_sbj,
            'intrg_mse_certain_obs':intrg_mse_certain_obs,

            'mse':get_mse_vs_theory(out_model),
            'descriptor':descriptor
        }
    )

    df = out_model['true_prob'].groupby(['subj','t']).apply(
        lambda x: (x.sort_values('g')['pred'].diff()> 0).any(),
        include_groups=False
    ).rename('any_violation').reset_index()
    df_fract_t_violated_per_subj = df.groupby('subj')['any_violation'].mean().reset_index()

    df = out_model['true_prob'].groupby(['subj','t']).apply(
        lambda x: (x.sort_values('g')['pred'].diff().max()),
        include_groups=False
    ).rename('max_violation').reset_index()
    df_fract_t_violated_per_subj['max_violation'] = df_fract_t_violated_per_subj['subj'].map(
        df.groupby('subj')['max_violation'].max()
    )
    row = df_fract_t_violated_per_subj['any_violation'].describe().to_dict()
    row['model_type'] = model_name
    row['run_id'] = run_id
    row['descriptor'] = descriptor
    model_to_prop_t_violated_per_subj.append(row)


    row = df_fract_t_violated_per_subj.loc[
        df_fract_t_violated_per_subj['any_violation'] > 0,
        'max_violation'
    ].describe().to_dict()
    row['model_type'] = model_name
    row['run_id'] = run_id
    model_to_max_violated_per_subj.append(row)
    print(f'finished processing {run_id}')

df_prop_t_violated_per_subj = pd.DataFrame(model_to_prop_t_violated_per_subj)
df_max_violated_per_subj = pd.DataFrame(model_to_max_violated_per_subj)
df_model_to_metric = pd.DataFrame(model_to_metric)
del model_to_prop_t_violated_per_subj
del model_to_max_violated_per_subj
del model_to_metric

### Visualize performance

In [ ]:
df_model_to_metric['median_prop_t_violated'] = df_model_to_metric['run_id'].map(
    df_prop_t_violated_per_subj.set_index('run_id')['50%']
)
df_model_to_metric['median_max_violation'] = df_model_to_metric['run_id'].map(
    df_max_violated_per_subj.set_index('run_id')['50%']
)
df_model_to_metric['max_prop_t_violated'] = df_model_to_metric['run_id'].map(
    df_prop_t_violated_per_subj.set_index('run_id')['max']
)
df_model_to_metric['max_max_violation'] = df_model_to_metric['run_id'].map(
    df_max_violated_per_subj.set_index('run_id')['max']
)

In [ ]:
df_model_to_metric

In [ ]:
for y in [
    'intrg_brier_ipcw_by_subj',
    'intrg_brier_ipcw_by_g',
    'intrg_brier_intrvl_imputed_ipcw_by_sbj',
    'intrg_brier_intrvl_imputed',
    'intrg_mse_certain_obs_ipcw_by_sbj',
    'intrg_mse_certain_obs',
    'mse'

]:
    fig, ax = plt.subplots(1,1, figsize=(6, 4))
    sns.boxplot(
        data=df_model_to_metric,
        y=y,
        x='model_type',
        hue='descriptor',
        ax=ax,
    )
    ax.legend(bbox_to_anchor=(1.8, 1), title='model_type')
    ax.set(ylabel='score', title=y)
    plt.grid()
    plt.xticks(rotation=20, ha='right')


In [ ]:
for y in [
    'intrg_brier_ipcw_by_subj',
    'intrg_brier_ipcw_by_g',
    'intrg_brier_intrvl_imputed_ipcw_by_sbj',
    'intrg_brier_intrvl_imputed',
    'intrg_mse_certain_obs_ipcw_by_sbj',
    'intrg_mse_certain_obs'

]:
    fig, ax = plt.subplots(1,1, figsize=(6, 4))
    sns.scatterplot(
        data=df_model_to_metric,
        x='mse',
        y=y,
        hue='model_type',
        style='descriptor',
        ax=ax,
    )
    ax.legend(bbox_to_anchor=(1.8, 1), title='model_type')

In [ ]:
for y in [
    'intrg_brier_ipcw_by_subj',
    'intrg_brier_ipcw_by_g',
    'intrg_brier_intrvl_imputed_ipcw_by_sbj',
    'intrg_brier_intrvl_imputed'

]:
    fig, ax = plt.subplots(1,1, figsize=(6, 4))
    sns.scatterplot(
        data=df_model_to_metric,
        x='intrg_mse_certain_obs',
        y=y,
        hue='model_type',
        style='descriptor',
        ax=ax,
    )
    ax.legend(bbox_to_anchor=(1.8, 1), title='model_type')

In [ ]:
for y in [
    'intrg_brier_ipcw_by_subj',
    'intrg_brier_ipcw_by_g',
    'intrg_brier_intrvl_imputed_ipcw_by_sbj',
    'intrg_brier_intrvl_imputed'

]:
    fig, ax = plt.subplots(1,1, figsize=(6, 4))
    sns.scatterplot(
        data=df_model_to_metric,
        x='intrg_mse_certain_obs_ipcw_by_sbj',
        y=y,
        hue='model_type',
        style='descriptor',
        ax=ax,
    )
    ax.legend(bbox_to_anchor=(1.8, 1), title='model_type')

In [ ]:
df_model_to_metric.groupby(['model_type', 'descriptor'])['mse'].describe()

# Clean up

In [ ]:
with torch.no_grad():
    torch.cuda.empty_cache()
gc.collect()